## Cell 0 - Install

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" accelerate bitsandbytes "pandas==2.2.2" "pillow==11.3.0" tqdm kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.8 MB/s eta 0:00:00


## Cell 1 — Imports + setup

In [2]:
import os, json, random, gc, shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU: True
Tesla T4


## Cell 2 — Download data

In [3]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [5]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
assert len(matches) > 0

DATA_ROOT = matches[0].parents[2]

print("DATA_ROOT:", DATA_ROOT)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions
DATA_ROOT: /root/.cache/kagglehub/competitions/pixels-to-predictions/images
Train: 3109 Val: 1048 Test: 1008


## Cell 3 — Mount Drive + save paths

In [6]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = Path("/content/lora_v8_dora_attnmlp_nativeres")
DRIVE_SAVE = Path("/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres")

Mounted at /content/drive


## Cell 4 — Image loader

In [7]:
IMAGE_CAP = 768  # if OOM, change to 512

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    if not path.exists():
        path = DATA_ROOT.parent / row["image_path"]

    img = Image.open(path).convert("RGB")
    img.thumbnail((IMAGE_CAP, IMAGE_CAP), Image.BICUBIC)
    return img


def img_384(row):
    path = DATA_ROOT / row["image_path"]
    if not path.exists():
        path = DATA_ROOT.parent / row["image_path"]

    img = Image.open(path).convert("RGB")
    img.thumbnail((384, 384), Image.BICUBIC)
    return img

## Cell 5 — Prompt builder

In [8]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_prompt(row, include_answer=False):
    choices = row["choices"]

    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {choice}"
        for i, choice in enumerate(choices)
    )

    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"

    if lecture:
        prompt += f"Lecture: {lecture}\n\n"

    if hint:
        prompt += f"Hint: {hint}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}. {choices[ans]}"

    return prompt


print(build_prompt(train_df.iloc[0], include_answer=True)[:1000])

<image>
You are solving a science multiple-choice question.
Use the image and text to choose the best answer.
Answer with ONLY the letter and the full text of the correct choice.

Lecture: Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive

## Cell 6 — Load model + DoRA LoRA config

In [9]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# IMPORTANT for training/masking
processor.tokenizer.padding_side = "right"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

lora_config = LoraConfig(
    r=6,
    lora_alpha=12,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", trainable)

assert trainable <= 5_000_000, "Trainable params exceed 5M!"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

trainable params: 3,892,224 || all params: 511,374,528 || trainable%: 0.7611
Trainable params: 3892224


## Cell 7 — Dataset + collator

In [10]:
class VQATrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = load_image(row)
        prompt_text = build_prompt(row, include_answer=False)
        full_text = build_prompt(row, include_answer=True)

        return {
            "image": image,
            "prompt_text": prompt_text,
            "full_text": full_text,
        }


def collate_train(batch):
    images = [x["image"] for x in batch]
    full_texts = [x["full_text"] for x in batch]
    prompt_texts = [x["prompt_text"] for x in batch]

    full_inputs = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )

    prompt_inputs = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )

    labels = full_inputs["input_ids"].clone()
    labels[:] = -100

    for i in range(len(batch)):
        full_len = int(full_inputs["attention_mask"][i].sum().item())
        prompt_len = int(prompt_inputs["attention_mask"][i].sum().item())

        labels[i, prompt_len:full_len] = full_inputs["input_ids"][i, prompt_len:full_len]

    full_inputs["labels"] = labels
    return full_inputs

## Cell 8 — Masking sanity check

In [11]:
processor.tokenizer.padding_side = "right"

tmp_ds = VQATrainDataset(train_df.head(4))
tmp_batch = collate_train([tmp_ds[i] for i in range(4)])

for i in range(4):
    labels = tmp_batch["labels"][i]
    visible = labels[labels != -100]
    print("Loss text:", repr(processor.tokenizer.decode(visible)))

Loss text: " C. the male's tadpoles will become adult frogs"
Loss text: " A. the female's offspring will live longer"
Loss text: " B. the lioness's cubs will survive attacks"
Loss text: " B. the gull's offspring will survive"


## Cell 9 — Smoke test training

In [11]:
SMOKE_STEPS = 200
TRAIN_FRAC_SMOKE = 0.10
GRAD_ACCUM = 16
LR = 3e-5

processor.tokenizer.padding_side = "right"

smoke_df = train_df.sample(frac=TRAIN_FRAC_SMOKE, random_state=SEED).reset_index(drop=True)

smoke_ds = VQATrainDataset(smoke_df)
smoke_loader = DataLoader(
    smoke_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_train,
    num_workers=0,
)

model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

losses = []
optimizer.zero_grad(set_to_none=True)

for step, batch in enumerate(tqdm(smoke_loader, desc="Smoke test")):
    if step >= SMOKE_STEPS:
        break

    batch = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }

    out = model(**batch)
    loss = out.loss / GRAD_ACCUM
    loss.backward()

    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    losses.append(out.loss.item())

    if (step + 1) % 32 == 0:
        print(f"Step {step+1}: avg32={np.mean(losses[-32:]):.4f}")

print("First 32 avg:", np.mean(losses[:32]))
print("Last 32 avg:", np.mean(losses[-32:]))

del optimizer, smoke_loader, smoke_ds
gc.collect()
torch.cuda.empty_cache()

Smoke test:   0%|          | 0/311 [00:00<?, ?it/s]

Step 32: avg32=0.5950
Step 64: avg32=0.5067
Step 96: avg32=0.4529
Step 128: avg32=0.3740
Step 160: avg32=0.3006
Step 192: avg32=0.3082
First 32 avg: 0.5949807161814533
Last 32 avg: 0.3346560660866089


## Cell 10 — Reload fresh model for full training

In [12]:
del model, base_model
gc.collect()
torch.cuda.empty_cache()

processor.tokenizer.padding_side = "right"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

model = get_peft_model(base_model, lora_config)

model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params:", trainable)
assert trainable <= 5_000_000

trainable params: 3,892,224 || all params: 511,374,528 || trainable%: 0.7611
Trainable params: 3892224


## Cell 11 — Full training

In [13]:
BATCH_SIZE_TRAIN = 1
GRAD_ACCUM = 16
EPOCHS = 1   # start with 1; continue to 2 only if val is promising
LR = 3e-5
WARMUP_RATIO = 0.05
CHECKPOINT_EVERY = 500

processor.tokenizer.padding_side = "right"

train_ds = VQATrainDataset(train_df)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,
    collate_fn=collate_train,
    num_workers=0,
)

total_steps = len(train_loader) * EPOCHS
opt_steps = max(1, total_steps // GRAD_ACCUM)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(opt_steps * WARMUP_RATIO)),
    num_training_steps=opt_steps,
)

loss_history = []
optimizer.zero_grad(set_to_none=True)
global_step = 0

for epoch in range(EPOCHS):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, batch in enumerate(pbar):
        batch = {
            k: v.to(model.device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        out = model(**batch)
        loss = out.loss / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        loss_history.append(out.loss.item())
        global_step += 1

        if global_step % 50 == 0:
            pbar.set_postfix(
                loss=f"{loss_history[-1]:.4f}",
                avg50=f"{np.mean(loss_history[-50:]):.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
            )

        if global_step % CHECKPOINT_EVERY == 0:
            ckpt_dir = Path(f"/content/drive/MyDrive/lora_v8_dora_ckpt_step_{global_step}")
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(ckpt_dir)
            processor.save_pretrained(ckpt_dir)
            print("Saved checkpoint:", ckpt_dir)

print("Training done")
print("First 50 avg:", np.mean(loss_history[:50]))
print("Last 50 avg:", np.mean(loss_history[-50:]))

Epoch 1/1:   0%|          | 0/3109 [00:00<?, ?it/s]

Saved checkpoint: /content/drive/MyDrive/lora_v8_dora_ckpt_step_500
Saved checkpoint: /content/drive/MyDrive/lora_v8_dora_ckpt_step_1000
Saved checkpoint: /content/drive/MyDrive/lora_v8_dora_ckpt_step_1500
Saved checkpoint: /content/drive/MyDrive/lora_v8_dora_ckpt_step_2000
Saved checkpoint: /content/drive/MyDrive/lora_v8_dora_ckpt_step_2500
Saved checkpoint: /content/drive/MyDrive/lora_v8_dora_ckpt_step_3000
Training done
First 50 avg: 0.7348761858046055
Last 50 avg: 0.07799187113894732


## Cell 12 — Save adapter

In [14]:
model.eval()
model.config.use_cache = True

SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

if DRIVE_SAVE.exists():
    shutil.rmtree(DRIVE_SAVE)

shutil.copytree(SAVE_DIR, DRIVE_SAVE)

print("Saved to:", DRIVE_SAVE)
print(list(DRIVE_SAVE.iterdir()))

Saved to: /content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres
[PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/chat_template.jinja'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/preprocessor_config.json'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/README.md'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/merges.txt'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/adapter_model.safetensors'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/adapter_config.json'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/vocab.json'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/special_tokens_map.json'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/added_tokens.json'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/processor_config.json'), PosixPath('/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres/tok

## Cell 13 — Inference helpers

In [15]:
processor.tokenizer.padding_side = "left"
model.eval()
model.config.use_cache = True

def get_candidate_token_ids(tokenizer):
    candidate_ids = {}

    for letter in CHOICE_LETTERS:
        forms = [letter, " " + letter, "\n" + letter, letter + ".", " " + letter + "."]
        ids = set()

        for form in forms:
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                ids.add(enc[-1])

        candidate_ids[letter] = sorted(ids)

    return candidate_ids


candidate_ids = get_candidate_token_ids(processor.tokenizer)


def predict_score_one(row):
    image = img_384(row)
    prompt = build_prompt(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]

    log_probs = torch.log_softmax(logits, dim=-1)

    scores = []
    for choice_idx in range(len(row["choices"])):
        letter = CHOICE_LETTERS[choice_idx]
        scores.append(log_probs[0, candidate_ids[letter]].max().item())

    pred = int(np.argmax(scores))
    return pred, scores

## Cell 14 — Full validation

In [16]:
val_preds = []
val_scores = []

for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Full val"):
    pred, scores = predict_score_one(row)
    val_preds.append(pred)
    val_scores.append(scores)

    torch.cuda.empty_cache()

val_results = val_df.copy().reset_index(drop=True)
val_results["pred"] = val_preds
val_results["scores"] = val_scores
val_results["correct"] = val_results["pred"] == val_results["answer"]

acc = val_results["correct"].mean()
print("DoRA Attn+MLP native-res full-val accuracy:", acc)

print("\nBy num_choices:")
display(val_results.groupby("num_choices")["correct"].agg(["mean", "count"]))

print("\nBy subject:")
display(val_results.groupby("subject")["correct"].agg(["mean", "count"]).sort_values("mean"))

val_results.to_csv("/content/lora_v8_dora_attnmlp_nativeres_val_results.csv", index=False)
val_results.to_csv("/content/drive/MyDrive/lora_v8_dora_attnmlp_nativeres_val_results.csv", index=False)

Full val:   0%|          | 0/1048 [00:00<?, ?it/s]

DoRA Attn+MLP native-res full-val accuracy: 0.7127862595419847

By num_choices:


,mean,count
num_choices,,
2,0.762295,244
3,0.675197,508
4,0.817460,252
5,0.272727,44



By subject:


,mean,count
subject,,
language science,0.535714,28
natural science,0.674389,777
social science,0.855967,243


## Cell 15 — Test inference

In [ ]:
test_preds = []
test_scores = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test"):
    pred, scores = predict_score_one(row)
    test_preds.append(pred)
    test_scores.append(scores)

    torch.cuda.empty_cache()

print("Predictions:", len(test_preds), "Test rows:", len(test_df))

## Cell 16 — Create submission

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": test_preds,
})

submission["answer"] = submission["answer"].astype(int)

assert list(submission.columns) == ["id", "answer"]
assert len(submission) == len(sample_submission)
assert submission["id"].tolist() == sample_submission["id"].tolist()
assert submission["answer"].notna().all()

for idx, row in test_df.iterrows():
    pred = int(submission.loc[idx, "answer"])
    assert 0 <= pred < len(row["choices"])

submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_lora_v8_dora_attnmlp_nativeres.csv", index=False)

print(submission.head())
print(submission["answer"].value_counts().sort_index())
print("Saved submission")

## Cell 17 — Download

In [ ]:
from google.colab import files
files.download("/content/submission.csv")